In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("abfss://bronze@energyprojectdl.dfs.core.windows.net/API ingestion/electricity_daily/ingest_date=2026-09-23")

In [0]:
display(df)

period,respondent,respondent-name,type,type-name,value,value-units
2026-09-23T00:00:00.000Z,CAL,California,D,Demand,38047,megawatthours
2026-09-23T00:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,17569,megawatthours
2026-09-23T00:00:00.000Z,PJM,"PJM Interconnection, LLC",D,Demand,95283,megawatthours
2026-09-23T00:00:00.000Z,TEX,Texas,D,Demand,81221,megawatthours
2026-09-22T23:00:00.000Z,CAL,California,D,Demand,37343,megawatthours
2026-09-22T23:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,17316,megawatthours
2026-09-22T23:00:00.000Z,PJM,"PJM Interconnection, LLC",D,Demand,94582,megawatthours
2026-09-22T23:00:00.000Z,TEX,Texas,D,Demand,84168,megawatthours
2026-09-22T22:00:00.000Z,CAL,California,D,Demand,37286,megawatthours
2026-09-22T22:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,16518,megawatthours


In [0]:
df = df.dropDuplicates(["period", "respondent", "type"])

In [0]:
display(df)

period,respondent,respondent-name,type,type-name,value,value-units
2026-09-23T00:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,17569,megawatthours
2026-09-22T17:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,13951,megawatthours
2026-09-22T16:00:00.000Z,PJM,"PJM Interconnection, LLC",D,Demand,93712,megawatthours
2026-09-22T15:00:00.000Z,CAL,California,D,Demand,32262,megawatthours
2026-09-22T15:00:00.000Z,PJM,"PJM Interconnection, LLC",D,Demand,93174,megawatthours
2026-09-22T10:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,13646,megawatthours
2026-09-22T10:00:00.000Z,PJM,"PJM Interconnection, LLC",D,Demand,80032,megawatthours
2026-09-22T03:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,15915,megawatthours
2026-09-22T00:00:00.000Z,TEX,Texas,D,Demand,81200,megawatthours
2026-09-22T23:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,17316,megawatthours


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_demo = df.dropDuplicates(["period", "respondent", "type"])
display(df_demo)

period,respondent,respondent-name,type,type-name,value,value-units
2026-09-23T00:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,17569,megawatthours
2026-09-22T17:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,13951,megawatthours
2026-09-22T16:00:00.000Z,PJM,"PJM Interconnection, LLC",D,Demand,93712,megawatthours
2026-09-22T15:00:00.000Z,CAL,California,D,Demand,32262,megawatthours
2026-09-22T15:00:00.000Z,PJM,"PJM Interconnection, LLC",D,Demand,93174,megawatthours
2026-09-22T10:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,13646,megawatthours
2026-09-22T10:00:00.000Z,PJM,"PJM Interconnection, LLC",D,Demand,80032,megawatthours
2026-09-22T03:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,15915,megawatthours
2026-09-22T00:00:00.000Z,TEX,Texas,D,Demand,81200,megawatthours
2026-09-22T23:00:00.000Z,NYIS,New York Independent System Operator,D,Demand,17316,megawatthours


In [0]:
df_demo = df

In [0]:
from pyspark.sql.functions import count as _count

In [0]:
df.groupBy("respondent").agg(_count("*").alias("row_count")).show()

+----------+---------+
|respondent|row_count|
+----------+---------+
|      NYIS|       25|
|       PJM|       25|
|       CAL|       25|
|       TEX|       25|
+----------+---------+



In [0]:
df.filter(col("value").isNull()).count()

0

In [0]:
df.filter(col("period").isNull()).count()

0

In [0]:
silver_path = 'abfss://silver@energyprojectdl.dfs.core.windows.net/data'

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .save(silver_path)


In [0]:
df_day2 = spark.read.format("csv")\
    .load("abfss://bronze@energyprojectdl.dfs.core.windows.net/API ingestion/electricity_daily")

display(df_day2)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,ingest_date
period,respondent,respondent-name,type,type-name,value,value-units,2026-09-23
2026-09-23T21,CAL,California,D,Demand,35590,megawatthours,2026-09-23
2026-09-23T21,NYIS,New York Independent System Operator,D,Demand,14540,megawatthours,2026-09-23
2026-09-23T21,PJM,"PJM Interconnection, LLC",D,Demand,90642,megawatthours,2026-09-23
2026-09-23T21,TEX,Texas,D,Demand,84893,megawatthours,2026-09-23
2026-09-23T20,CAL,California,D,Demand,38630,megawatthours,2026-09-23
2026-09-23T20,NYIS,New York Independent System Operator,D,Demand,13543,megawatthours,2026-09-23
2026-09-23T20,PJM,"PJM Interconnection, LLC",D,Demand,89415,megawatthours,2026-09-23
2026-09-23T20,TEX,Texas,D,Demand,83291,megawatthours,2026-09-23
2026-09-23T19,CAL,California,D,Demand,38936,megawatthours,2026-09-23
